<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Neural Collaborative Filtering

Neural Collaborative Filtering (NCF) is a well known recommendation algorithm that generalizes the matrix factorization problem with multi-layer perceptron. 

This notebook provides an example of how to utilize and evaluate NCF implementation in the `recommenders`. We use a smaller dataset in this example to run NCF efficiently with GPU acceleration.

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import sys
import pandas as pd
import tensorflow as tf
# tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.utils.timer import Timer
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.datasets.python_splitters import python_random_split
from recommenders.evaluation.python_evaluation import (
    map, ndcg_at_k, precision_at_k, recall_at_k
)
from recommenders.utils.notebook_utils import store_metadata

from datasets import outfits

print("System version: {}".format(sys.version))
print("Pandas version: {}".format(pd.__version__))
print("Tensorflow version: {}".format(tf.__version__))

System version: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12) 
[GCC 13.3.0]
Pandas version: 2.3.3
Tensorflow version: 2.20.0


Set the default parameters.

In [17]:
# top k items to recommend
TOP_K = 10

# Change data size as appropriate
OUTFITS_DATA_SIZE = '500'

# Model parameters
EPOCHS = 100
BATCH_SIZE = 256

SEED = 42

### 1. Download the dataset

In [ ]:
df = outfits.load_pandas_df(
    header=["UserId", "Weather", "Season", "Event", "Clothing", "Rating"],
    filepath=f"datasets/csv/synthetic_data_feature2.csv"
)

df['ClothingId'] = df['Clothing'].astype('category').cat.codes
print(df)

     UserId Weather  Season     Event           Clothing  Rating  ClothingId
0         1   Humid  Summer     Party  Button-down shirt     2.5           1
1         1  Cloudy    Fall       Gym         Sweatshirt     3.3          13
2         1   Humid  Spring  Business  Button-down shirt     3.5           1
3         1   Windy  Spring      Home  Long-sleeve shirt     3.9          10
4         1   Sunny  Summer     Beach            T-shirt     3.2          14
..      ...     ...     ...       ...                ...     ...         ...
495      50   Sunny  Spring       Gym            Joggers     2.2           9
496      50  Cloudy    Fall    Hiking  Long-sleeve shirt     4.0          10
497      50   Rainy    Fall     Beach           Tank top     1.4          15
498      50   Snowy  Winter    Travel            Joggers     3.5           9
499      50   Windy    Fall      Date             Henley     3.5           6

[500 rows x 7 columns]


### 2. Split the data using the Spark splitter provided in utilities

In [19]:
train, test = python_random_split(
    df, 
    ratio=0.75
)
train = train[train['Rating'] > 0]
assert len(train) > 0, "STOP: Training set is empty immediately after splitting."

Filter out any users or items in the test set that do not appear in the training set.

In [20]:
test = test[test["UserId"].isin(train["UserId"].unique())]
test = test[test["ClothingId"].isin(train["ClothingId"].unique())]

train_sorted = train.sort_values(by="UserId")
test_sorted = test.sort_values(by="UserId")

Write datasets to csv files.

In [21]:
train_file = "./train.csv"
test_file = "./test.csv"
train_sorted.to_csv(train_file, index=False)
test_sorted.to_csv(test_file, index=False)

Generate an NCF dataset object from the data subsets.

In [22]:
data = NCFDataset(
  train_file=train_file, 
  test_file=test_file, 
  seed=SEED, 
  col_user="UserId", 
  col_item="ClothingId", 
  col_rating="Rating")

assert data.n_users > 0 and data.n_items > 0, "STOP: The NCFDataset object loaded no users or items."

INFO:recommenders.models.ncf.dataset:Indexing ./train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing ./test.csv ...
INFO:recommenders.models.ncf.dataset:Indexing ./test_full.csv ...


### 3. Train the NCF model on the training data, and get the top-k recommendations for our testing data

NCF accepts implicit feedback and generates prospensity of items to be recommended to users in the scale of 0 to 1. A recommended item list can then be generated based on the scores. Note that this quickstart notebook is using a smaller number of epochs to reduce time for training. As a consequence, the model performance will be slighlty deteriorated. 

In [23]:
model = NCF (
    n_users=data.n_users, 
    n_items=data.n_items,
    model_type="NeuMF",
    n_factors=4,
    layer_sizes=[16,8,4],
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=1e-3,
    verbose=10,
    seed=SEED
)

/users/fshodipe/.conda/envs/daily_outfit/lib/python3.9/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
2025-11-16 13:40:04.496099: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1763318404.505434  367828 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


In [24]:
# Available for use with larger datasets
with Timer() as train_time:
    model.fit(data)

print("Took {} seconds for training.".format(train_time))

INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [0.03s]: train_loss = 0.685446 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 20 [0.03s]: train_loss = 0.676594 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 30 [0.03s]: train_loss = 0.662951 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 40 [0.03s]: train_loss = 0.647925 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 50 [0.03s]: train_loss = 0.632453 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 60 [0.03s]: train_loss = 0.617040 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 70 [0.03s]: train_loss = 0.594038 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 80 [0.03s]: train_loss = 0.577089 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 90 [0.03s]: train_loss = 0.550325 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 100 [0.03s]: train_loss = 0.530882 


Took 3.2444 seconds for training.


In [25]:
with Timer() as test_time:
    users, items, preds = [], [], []
    item = list(train.ClothingId.unique())
    for user in train.UserId.unique():
        user = [user] * len(item) 
        users.extend(user)
        items.extend(item)
        preds.extend(list(model.predict(user, item, is_list=True)))

    all_predictions = pd.DataFrame(data={"UserId": users, "ClothingId":items, "prediction":preds})

    merged = pd.merge(train, all_predictions, on=["UserId", "ClothingId"], how="outer")
    all_predictions = merged[merged.Rating.isnull()].drop('Rating', axis=1)

print("Took {} seconds for prediction.".format(test_time))

Took 0.0339 seconds for prediction.


### 4. Evaluate how well NCF performs

The ranking metrics are used for evaluation.

In [26]:
eval_map = map(test, all_predictions, col_prediction='prediction', k=TOP_K, col_user='UserId', col_item='ClothingId')
eval_ndcg = ndcg_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K, col_user='UserId', col_item='ClothingId', col_rating='Rating')
eval_precision = precision_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K, col_user='UserId', col_item='ClothingId')
eval_recall = recall_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K, col_user='UserId', col_item='ClothingId')

print("MAP:\t%f" % eval_map,
      "NDCG:\t%f" % eval_ndcg,
      "Precision@K:\t%f" % eval_precision,
      "Recall@K:\t%f" % eval_recall, sep='\n')

MAP:	0.159193
NDCG:	0.294703
Precision@K:	0.122917
Recall@K:	0.525347


In [27]:
# Record results for tests - ignore this cell
store_metadata("map", eval_map)
store_metadata("ndcg", eval_ndcg)
store_metadata("precision", eval_precision)
store_metadata("recall", eval_recall)
store_metadata("train_time", train_time.interval)
store_metadata("test_time", test_time.interval)